# 🔧 Verificação de ambiente — 2 minutos

### Tutorial: *Rumo ao Desconhecido — Tratando Drift em Machine Learning*
**Python Brasil 2026 · 3h30**

---

## Faça isto **em casa**, antes do evento

Este notebook não ensina nada. Ele só responde a uma pergunta:
**seu ambiente vai funcionar no dia?**

1. Rode a célula abaixo (menu *Ambiente de execução → Executar tudo*, ou `Ctrl+F9`)
2. Espere ~90 segundos
3. Se aparecer **✅ AMBIENTE PRONTO**, feche e não pense mais nisso
4. Se aparecer qualquer ❌, me mande um print por e-mail

> **Não precisa instalar nada** na sua máquina. Não precisa baixar os 14 GB
> da Kaggle. Não precisa de conta paga.

---

### ⚠️ Se você usar uma conta institucional

Algumas universidades e empresas **bloqueiam o Google Colab** por política de
TI. Se a célula abaixo nem começar a rodar, abra este link numa **janela
anônima** com uma **conta Google pessoal**. Esse é, de longe, o problema mais
comum — e o único que não dá para resolver no dia.

In [ ]:
# ============================================================
#  CÉLULA 0 — SETUP
#  Idempotente: pode rodar quantas vezes quiser.
#  Se o runtime do Colab cair durante o tutorial, rode isto de novo.
# ============================================================
!curl -sSL https://raw.githubusercontent.com/arcursino/python-br-2026/main/setup_colab.py -o /tmp/setup_colab.py
%run /tmp/setup_colab.py

### 🟡 Se aparecerem avisos (⚠️) em vez de ❌

O setup instala em **três camadas**, e só a primeira é obrigatória:

| camada | conteúdo | se falhar |
|---|---|---|
| 1 · núcleo + dev | numpy, pandas, scipy, sklearn, pytest | ⛔ aborta — chame o monitor |
| 2 · `river` | detectores sequenciais (Bloco IV) | ⚠️ segue; a célula degrada sozinha |
| 3 · `evidently` | validação cruzada (Bloco II-A) | ⚠️ segue; usamos a nossa implementação |

Ou seja: **`✅ AMBIENTE PRONTO — com degradações aceitáveis` é um resultado bom.**
Você perde no máximo uma demonstração de 3 minutos, e nada dos exercícios.

> O `alibi-detect` **não** é instalado de propósito: em Python 3.12+ ele
> quebra na geração de metadata e derrubava o setup inteiro. Fica em
> `pip install ".[mercado-alibi]"` para quem quiser tentar num venv com 3.11.
> Isso, por si só, já é um argumento sobre dependências em MLOps.

---
## Teste de fumaça 1 — a tese do tutorial em duas linhas

Se a célula acima ficou verde, esta deve rodar em ~15 segundos.

In [ ]:
import numpy as np
from scipy.stats import ks_2samp
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(0)
X_tr = rng.normal(0, 1, (4000, 2))
y_tr = (X_tr[:, 0] + X_tr[:, 1] > 0).astype(int)
m = LogisticRegression().fit(X_tr, y_tr)

# (A) DATA DRIFT — P(X) muda muito, P(y|X) intacto
X_a = rng.normal(1.5, 1.6, (4000, 2))
y_a = (X_a[:, 0] + X_a[:, 1] > 0).astype(int)

# (B) CONCEPT DRIFT — P(X) idêntico, P(y|X) invertido
X_b = rng.normal(0, 1, (4000, 2))
y_b = (X_b[:, 0] - X_b[:, 1] > 0).astype(int)

for nome, Xn, yn in [("A) data drift   ", X_a, y_a), ("B) concept drift", X_b, y_b)]:
    ks = max(ks_2samp(X_tr[:, j], Xn[:, j]).statistic for j in range(2))
    auc = roc_auc_score(yn, m.predict_proba(Xn)[:, 1])
    print(f"{nome} | KS máx = {ks:.3f} | AUC = {auc:.3f}")

print("\nA: o detector de distribuição GRITA e o modelo está intacto.")
print("B: o detector de distribuição fica MUDO e o modelo virou moeda.")

# Honestidade sobre o brinquedo — e o gancho para o Bloco III.
#
# Aqui P(X) é IDÊNTICO por construção (mesmo rng, mesmos parâmetros): só o
# rótulo inverteu de x0+x1 para x0-x1. A mudez é PERFEITA, e é perfeita porque
# este é um brinquedo.
#
# Na fixture de fábrica o caso é pior. O TC-3 (transdutor descalibrado) deixa
# um resíduo em P(X): a compensação do operador cancela 87.5% do offset, não
# 100%. O detector CALIBRADO mede efeito 0.018 com p ~ 1e-6 — sinal pequeno,
# real e estatisticamente inegável. Enquanto isso o TC-2, que é drift benigno
# de campanha, mede 1.91: cem vezes MAIOR.
#
# Sinal pequeno e real é mais perigoso que sinal nenhum, porque passa por
# ruído. E a ordenação por magnitude em P(X) é a INVERSA da ordenação por
# urgência.
print("\nNo toy, P(X) é idêntico por construção — mudez PERFEITA.")
print("Na fábrica é pior: P(X) SUSSURRA. Bloco III, TC-3, efeito 0.018 (p~1e-6).")
print("E o TC-2, que é BENIGNO, mede 1.91 — cem vezes maior que o grave.")
print("Sinal pequeno e real engana mais que sinal nenhum: ele passa por ruído.")
print("\nTodo o tutorial é consequência dessas duas linhas.")

---
## Teste de fumaça 2 — o limiar que você herdou

Se esta célula rodar, o seu `driftkit` está na versão certa. E de brinde ela
responde a uma pergunta que provavelmente ninguém te fez:

> **de onde vem o `PSI > 0.1`?**

In [ ]:
from driftkit.detectors import n_equivalente, piso_analitico

print(f"o limiar 0.1, com 10 bins, é o piso de RUÍDO de uma janela de "
      f"~{n_equivalente(0.10):.0f} observações.\n")

print(f"  {'n por janela':>14}  {'piso real':>10}   o 0.1 está...")
print("  " + "-" * 48)
for n in (340, 2_700, 40_000, 500_000):
    p = piso_analitico(n, n)
    print(f"  {n:>14,}  {p:>10.6f}   {0.10 / p:>6.0f}x solto")

print(
    "\n>>> Um monitor com limiar FIXO fica progressivamente mais CEGO\n"
    ">>> conforme o seu volume de dados cresce. Você paga por mais dados\n"
    ">>> e joga fora todo o poder estatístico que eles compram.\n"
    ">>> É o Bloco III inteiro. Guarde a dúvida para o dia."
)

---
## Teste de fumaça 3 — a CLI e o exit code

Estes dois comandos são o entregável do tutorial. Se rodarem aqui, rodam no dia.

In [ ]:
from driftkit.notebook import cli

# calibra o detector E diagnostica se a referência merece confiança
cli("calibrar", "--seed", "7", "--repeticoes", "20", "--modo", "ambos")

In [ ]:
cli("simular", "--de", "40", "--ate", "130", "--passo", "10")

# Leia a coluna `exit`:
#    0  nada a fazer
#   10  retreino aprovado
#   20  retreino BLOQUEADO  ← o tutorial inteiro em um número
#   30  dados insuficientes

---
## Pronto?

| resultado | o que fazer |
|---|---|
| ✅ tudo verde | Nada. Nos vemos no dia. |
| ✅ verde **com degradações** (`river`/`evidently` ausentes) | Também está pronto. Siga. |
| ⚠️ avisos de versão | Provavelmente ok. Traga o print. |
| ❌ `PENDÊNCIAS QUE IMPEDEM O TUTORIAL` | Me mande o print por e-mail **antes** do evento. |

Se der ❌, o próprio setup já imprime o diagnóstico e o comando de desbloqueio.
Cole a saída inteira no e-mail — inclusive a versão do Python que ele mostra na
primeira linha.

### No dia, leve

- Sua conta Google **testada** neste notebook
- Disposição para trabalhar **em dupla** (é escolha pedagógica, não falta de máquina)

### Não leve

- Ansiedade com setup. Se algo quebrar no dia, há plano B, C e D —
  incluindo os notebooks já executados em `notebooks/executados/`,
  para acompanhar lendo as saídas reais.

---

📦 `github.com/arcursino/python-br-2026`